# DeepWeeds Lab Day 2 — runnable pipeline

Run one stage at a time. Backbone and recipe selection, plus temperature fitting, use **validation only**. The final test cell is disabled by default and must be run once per seed after you freeze the final recipe and inference method.

Outputs are written under the repository: `runs/`, `runs_final/`, `predictions/`, `curves/`, `eda/`, and `results.xlsx`. Put the clone on persistent Drive storage (Colab) or `/kaggle/working` (Kaggle) before starting.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json

def find_repo(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'eval.py').is_file() and (candidate / 'starter').is_dir():
            return candidate
    raise FileNotFoundError('Clone K4-Track4-Day2-Deeplearning-Advance into persistent storage first.')

REPO_DIR = find_repo()
os.chdir(REPO_DIR)
print('Repo:', REPO_DIR)
print('GPU runtime must be enabled before training.')

## 1. Install dependencies and prepare data

Colab/Kaggle normally provide the matching GPU-enabled `torch` and `torchvision`. The requirements file installs the remaining fixed dependencies. `prepare_data.py` checks the provided zip's MD5, extracts the 17,509 JPEGs, and downloads the author's `labels.csv` and fold-0 split files if missing. Internet is needed for labels, pretrained ImageNet weights, and for downloading `images.zip` if it is not already in the repo.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements.txt')], check=True)
import torch, torchvision, timm, pandas, numpy
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| timm', timm.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU accelerator in Colab/Kaggle before training.')

In [ ]:
subprocess.run([sys.executable, 'prepare_data.py'], check=True)
assert len(list(Path('data/images').glob('*.jpg'))) == 17509
print('Image count and fold-0 CSVs are ready.')

## 2. Fold checks, EDA, and pipeline smoke checks

This reports split and class counts, verifies empty overlaps and all image paths, saves sample/augmentation plots, checks nine-class CE and focal `gamma=0`, and tries to overfit one small batch. Fix any failure before the long runs.

In [ ]:
subprocess.run([sys.executable, 'starter/eda.py', '--smoke-test', '--device', 'cuda'], check=True)
print('EDA evidence: eda/class_distribution.png, eda/train_samples.png, eda/augmentation_samples.png, eda/pipeline_smoke.json')

## 3. Backbone sweep (validation only)

Runs ResNet-50, ResNeXt-50, ConvNeXt-Tiny, DeiT-Small, and MobileNetV3-Large with the same T00 recipe, seed, and epoch count. Checkpoints, logs, and one curve per seed are saved. If GPU time is limited, lower epochs consistently for this sweep and write that choice in the report.

In [ ]:
EPOCHS = 12
BATCH_SIZE = 64
subprocess.run([sys.executable, 'starter/run_experiments.py', 'backbones', '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE)], check=True)

In [ ]:
backbone_summaries = []
for path in sorted(Path('runs').glob('B*/seed0/summary.json')):
    row = json.loads(path.read_text(encoding='utf-8'))
    backbone_summaries.append({k: row.get(k) for k in ('exp_id','backbone','weight_tag','num_params_m','gmac','macro_f1_val','top1_val','mean_epoch_seconds')})
import pandas as pd
pd.DataFrame(backbone_summaries).sort_values('macro_f1_val', ascending=False)

## 4. Controlled training recipes (validation only)

Choose one backbone from the table above using validation macro-F1, parameter count, and latency. The plan tests initialization, augmentation, loss, sampler, Mixup/CutMix, EMA, and one combined recipe. Each individual ablation changes one T00 factor; T12 is the explicit combination.

In [ ]:
SELECTED_BACKBONE = 'resnet50'  # Replace using validation results above.
subprocess.run([sys.executable, 'starter/run_experiments.py', 'ablations', '--backbone', SELECTED_BACKBONE, '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE)], check=True)

In [ ]:
recipe_summaries = []
for path in sorted(Path('runs').glob('T*/seed0/summary.json')):
    row = json.loads(path.read_text(encoding='utf-8'))
    recipe_summaries.append({k: row.get(k) for k in ('exp_id','backbone','init','best_epoch','macro_f1_val','top1_val','ece_val','mean_epoch_seconds')})
pd.DataFrame(recipe_summaries).sort_values('macro_f1_val', ascending=False)

## 5. Compare inference methods on validation

Select a trained recipe using validation results. This compares 1-view, horizontal-flip probability/logit averaging, five-crop, supported multi-scale, temperature scaling, BatchNorm fusion, and FP16 latency where available. Timing excludes input transforms, warms up, synchronizes CUDA, and reports p50/p95/p99. The multi-scale row can be unsupported for fixed-resolution transformers.

In [ ]:
SELECTED_RUN = 'runs/T00/seed0'  # Replace with the trained recipe/checkpoint you want to compare.
subprocess.run([sys.executable, 'starter/inference_eval.py', 'compare', '--run-dir', SELECTED_RUN, '--out', 'inference_results.csv', '--device', 'cuda', '--iters', '50'], check=True)

## 6. Final retraining — val only

Set `SELECTED_CONFIG` to the config file of the recipe chosen from validation. This trains the selected recipe and T00 baseline with three seeds, stores checkpoints and validation predictions, and does not access test images for inference. Do not proceed until the recipe and inference method are fixed.

In [ ]:
SELECTED_CONFIG = 'runs/T05_label_smoothing/seed0/config.json'  # Replace with your val-selected config.
FINAL_ID = 'F01'
SEEDS = ('0', '1', '2')
subprocess.run([sys.executable, 'starter/run_final.py', '--selected-run', SELECTED_CONFIG, '--final-id', FINAL_ID, '--seeds', *SEEDS], check=True)

## 7. Freeze inference choice, then run final test exactly once per seed

First run the selected method on final validation checkpoints and inspect the results. Then set `RUN_TEST_ONCE = True` in the next cell and execute it once. The helper refuses to overwrite an existing test prediction. For temperature scaling, use `<base-method>-temp` (for example `oneview-temp`); T is fitted independently on each seed's validation logits.

In [ ]:
FINAL_METHOD = 'oneview-temp'  # Choose using validation only: oneview, hflip-prob, hflip-logit, fivecrop, multiscale, bn-fused, fp16; optional -temp.
for seed in (0, 1, 2):
    run_dir = f'runs_final/{FINAL_ID}/seed{seed}'
    subprocess.run([sys.executable, 'starter/inference_eval.py', 'final', '--run-dir', run_dir, '--split', 'val', '--method', FINAL_METHOD, '--exp-id', FINAL_ID], check=True)

In [ ]:
RUN_TEST_ONCE = False  # Change to True only after the final method/configuration is frozen.
if RUN_TEST_ONCE:
    planned_test_files = [Path(f'predictions/{exp}_seed{seed}_test.csv') for exp in ('F01', 'T00') for seed in (0, 1, 2)]
    existing = [str(path) for path in planned_test_files if path.exists()]
    if existing:
        raise FileExistsError(f'Test predictions already exist; refusing to repeat test: {existing}')
    for seed in (0, 1, 2):
        run_dir = f'runs_final/{FINAL_ID}/seed{seed}'
        subprocess.run([sys.executable, 'starter/inference_eval.py', 'final', '--run-dir', run_dir, '--split', 'test', '--method', FINAL_METHOD, '--exp-id', FINAL_ID], check=True)
        subprocess.run([sys.executable, 'starter/inference_eval.py', 'final', '--run-dir', f'runs_final/T00/seed{seed}', '--split', 'test', '--method', 'oneview', '--exp-id', 'T00'], check=True)

## 8. Official score, rubric estimate, and workbook

Run these after the final test predictions exist. `eval.py` is the scoring source of truth. Keep its source file unchanged. The workbook builder reads run logs, validation/test predictions, and inference measurements; all metrics must match the official scorer.

In [ ]:
for exp_id in ('F01', 'T00'):
    subprocess.run([sys.executable, 'eval.py', 'score', '--pred', f'predictions/{exp_id}_seed*_test.csv', '--test-csv', 'data/labels/test_subset0.csv', '--labels', 'data/labels/labels.csv', '--tag', exp_id, '--out', 'eval_out'], check=True)
subprocess.run([sys.executable, 'eval.py', 'grade', '--final', 'predictions/F01_seed*_test.csv', '--baseline', 'predictions/T00_seed*_test.csv', '--uncal', 'predictions/F01_uncal_seed*_test.csv', '--final-val', 'predictions/F01_seed*_val.csv', '--test-csv', 'data/labels/test_subset0.csv', '--labels', 'data/labels/labels.csv', '--out', 'eval_out'], check=True)
subprocess.run([sys.executable, 'starter/reporting.py', '--runs', 'runs', 'runs_final', '--predictions', 'predictions', '--labels', 'data/labels/labels.csv', '--test-csv', 'data/labels/test_subset0.csv', '--inference-csv', 'inference_results.csv', '--output', 'results.xlsx', '--curves', 'curves'], check=True)

### Saved artifacts

- `runs/<exp_id>/seed<k>/`: config, per-epoch CSV, best/last checkpoints, validation logits and summary.
- `runs_final/`: final and T00 retraining runs.
- `curves/`: one train/validation curve per run, plus EDA, trade-off, and final confusion matrix.
- `predictions/`: final selected-method and T00 test CSVs, final validation CSVs, optional uncalibrated test CSVs.
- `inference_results.csv`, `results.xlsx`, `eval_out/`: method comparisons and scored outputs.

Do not commit `data/`, checkpoints, or `runs/`; do commit your small code, plots, workbook, report, and required prediction CSVs.